In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import os

import pandas as pd
import torch
import torch.nn as nn

from src.dataset import make_loaders
from src.evaluate import predict_loader, summarize
from src.model import CNNBaseline
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()

## Controlled ablations: 

In [6]:
def save_result(row):
    new = pd.DataFrame([row])
    if os.path.exists(RESULTS):
        old = pd.read_csv(RESULTS)
        new = pd.concat([old[old['experiment'] != row['experiment']], new])
    new.to_csv(RESULTS, index=False)

def run_experiment(name, augment=False, dropout_p=0.0, pool_type='max',
                   weight_decay=0.0, scheduler_name='none', lr=1e-3, 
                   seed=SEED, epochs=20): 
    set_seed(seed)
    train_ds, val_ds, train_loader, val_loader = make_loaders(
        DATA, IMG_SIZE, BATCH, seed, augment)
    model = CNNBaseline(dropout_p=dropout_p, pool_type=pool_type).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler, step_on_val = make_scheduler(scheduler_name, optimizer)
    criterion = nn.CrossEntropyLoss()

    config = {
        'augment': augment, 'dropout_p': dropout_p, 'pool_type': pool_type,
        'weight_decay': weight_decay, 'scheduler': scheduler_name, 'lr': lr,
        'epochs': epochs,   
        'batch_size': BATCH, 'optimizer': 'AdamW'
    }
    meta = {
        'architecture': 'CNNBaseline',
        'class_to_idx': train_ds.class_to_idx,
        'img_size': IMG_SIZE, 'seed': seed, 'config': config
    }
    ckpt = f'../checkpoints/{name}.pt'
    history = fit(
        model, train_loader, val_loader, optimizer, criterion, device,
        epochs,  
        ckpt, f'../logs/{name}.csv',
        meta=meta, scheduler=scheduler, step_on_val=step_on_val
    )
    model.load_state_dict(torch.load(ckpt, weights_only=False)['model_state_dict'])
    trues, preds = predict_loader(model, val_loader, device)
    row = summarize(trues, preds, train_ds.classes)
    best = max(history, key=lambda r: r['val_macro_f1'])
    row.update({
        'experiment': name, 'best_epoch': best['epoch'],
        'train_loss': best['train_loss'], 'val_loss': best['val_loss'],
        'gap': best['val_loss'] - best['train_loss'],
        'final_lr': history[-1]['lr'], **config
    })
    save_result(row)
    return row

In [3]:
run_experiment('baseline')

{'epoch': 1, 'train_loss': 1.9553413287318318, 'val_loss': 1.8444182142034753, 'val_macro_f1': 0.22423817637990032, 'lr': 0.001}
{'epoch': 2, 'train_loss': 1.8017693628654454, 'val_loss': 1.7731077503848385, 'val_macro_f1': 0.27124182511477596, 'lr': 0.001}
{'epoch': 3, 'train_loss': 1.720818733843679, 'val_loss': 1.7174667943607678, 'val_macro_f1': 0.3227424310680683, 'lr': 0.001}
{'epoch': 4, 'train_loss': 1.7071282378724058, 'val_loss': 1.6363443157889626, 'val_macro_f1': 0.33467106678915515, 'lr': 0.001}
{'epoch': 5, 'train_loss': 1.6185846103329873, 'val_loss': 1.579552410175274, 'val_macro_f1': 0.3535665524610164, 'lr': 0.001}
{'epoch': 6, 'train_loss': 1.5660372438021803, 'val_loss': 1.5173494670298193, 'val_macro_f1': 0.3984368331651224, 'lr': 0.001}
{'epoch': 7, 'train_loss': 1.486762671287839, 'val_loss': 1.479341375672972, 'val_macro_f1': 0.4366946545831887, 'lr': 0.001}
{'epoch': 8, 'train_loss': 1.420507591060418, 'val_loss': 1.5206851878723542, 'val_macro_f1': 0.398951324

{'macro_precision': np.float64(0.6679023297987383),
 'macro_recall': np.float64(0.650526646183941),
 'macro_f1': np.float64(0.6477766872825036),
 'lowest_recall_class': 'kamyunet',
 'lowest_precision_class': 'kamyunet',
 'recall_ambulance': np.float64(0.7954545454545454),
 'precision_ambulance': np.float64(0.603448275862069),
 'recall_autobus': np.float64(0.6530612244897959),
 'precision_autobus': np.float64(0.5714285714285714),
 'recall_kamyun': np.float64(0.62),
 'precision_kamyun': np.float64(0.6326530612244898),
 'recall_kamyunet': np.float64(0.44680851063829785),
 'precision_kamyunet': np.float64(0.4883720930232558),
 'recall_minibus': np.float64(0.4888888888888889),
 'precision_minibus': np.float64(0.9166666666666666),
 'recall_savari': np.float64(0.76),
 'precision_savari': np.float64(0.6333333333333333),
 'recall_taxi': np.float64(0.88),
 'precision_taxi': np.float64(0.8461538461538461),
 'recall_vanet': np.float64(0.56),
 'precision_vanet': np.float64(0.6511627906976745),
 'ex

In [ ]:
run_experiment('aug' , augment= True)

{'epoch': 1, 'train_loss': 1.988957766668606, 'val_loss': 1.864803376755157, 'val_macro_f1': 0.20113252458857434, 'lr': 0.001}
{'epoch': 2, 'train_loss': 1.847071887147047, 'val_loss': 1.8157169134585889, 'val_macro_f1': 0.20476893205144442, 'lr': 0.001}
{'epoch': 3, 'train_loss': 1.7825682345267004, 'val_loss': 1.8007066717395535, 'val_macro_f1': 0.25742653892146733, 'lr': 0.001}
{'epoch': 4, 'train_loss': 1.7570005483329878, 'val_loss': 1.6952726534434728, 'val_macro_f1': 0.3198431148495691, 'lr': 0.001}
